# MISCFilter: Motion Blur Image Restoration (Kaggle/Colab Version)

This notebook provides a complete and memory-efficient pipeline to restore motion-blurred images using the **MISCFilter** (CVPR 2024). 

To handle large images (like your $8000 \times 5300$ images) without running out of GPU memory (OOM), this notebook utilizes a custom inference script (`inference_custom.py`) that partitions the image into $256 \times 256$ windows, processes them in small chunks (e.g. 8 at a time), and stitches them back together seamlessly.

## Step 1: Environment Setup & Directory Check
If you uploaded this notebook to Kaggle/Colab without cloning your GitHub repository first, this cell will clone your repo and change directory into it automatically.

In [ ]:
import os

# Update with your specific github repository URL if needed
REPO_URL = "https://github.com/ChengxuLiu/MISCFilter.git"

if not os.path.exists('models') and not os.path.exists('utils'):
    print("Cloning repository...")
    !git clone {REPO_URL}
    repo_name = REPO_URL.split('/')[-1].replace('.git', '')
    if os.path.exists(repo_name):
        %cd {repo_name}
else:
    print("Already inside or next to the repository folder.")
    if os.path.basename(os.getcwd()) != 'MISCFilter' and os.path.exists('MISCFilter'):
        %cd MISCFilter

print(f"Current working directory: {os.getcwd()}")

## Step 2: Install Dependencies & CuPy
We need to install PyTorch-dependent packages (like `kornia`, `ptflops`, `yacs`, `natsort`) and compile the warm-up learning rate scheduler. 

Additionally, **CuPy** is required to compile and run the custom CUDA filters in MISCFilter dynamically. We will detect your runtime's CUDA version and install the matching CuPy package.

In [ ]:
import torch
import sys

# 1. Install pip dependencies
print("Installing requirements...")
!pip install yacs joblib natsort kornia ptflops gdown opencv-python scikit-image matplotlib

# 2. Install PyTorch Gradual Warmup Scheduler
if os.path.exists('pytorch-gradual-warmup-lr'):
    print("Installing warmup scheduler...")
    !pip install ./pytorch-gradual-warmup-lr
else:
    !pip install git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git

# 3. Install matching CuPy package to enable CUDA filters
cuda_version = torch.version.cuda
print(f"Detected CUDA version in PyTorch: {cuda_version}")
if cuda_version:
    major = cuda_version.split('.')[0]
    if major == '12':
        print("Installing cupy-cuda12x...")
        !pip install cupy-cuda12x
    elif major == '11':
        print("Installing cupy-cuda11x...")
        !pip install cupy-cuda11x
    else:
        print("No pre-compiled cupy binary found for this version. Installing general cupy (takes longer to compile)...")
        !pip install cupy
else:
    print("Error: No CUDA detected. Please change your Kaggle/Colab runtime to GPU.")

## Step 3: Setup Pretrained Weights
We will check if weight checkpoints are already present in the directory (because they were cloned from your GitHub). If they are not found, we will download them from the official Google Drive.

In [ ]:
# Define the weight file we want to use
WEIGHT_FILE = "RealBlur_J.pth"
WEIGHTS_PATH = ""

if os.path.exists(WEIGHT_FILE):
    WEIGHTS_PATH = f"./{WEIGHT_FILE}"
    print(f"Found weights '{WEIGHT_FILE}' in the root directory! Skipping download.")
elif os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
    WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    print(f"Found weights '{WEIGHT_FILE}' in checkpoints directory! Skipping download.")
elif os.path.exists("checkpoints") and len(os.listdir("checkpoints")) > 0:
    # Check if there's any fallback weight file in checkpoints
    available_weights = [f for f in os.listdir("checkpoints") if f.endswith('.pth')]
    if available_weights:
        WEIGHTS_PATH = f"./checkpoints/{available_weights[0]}"
        print(f"Found weights '{available_weights[0]}' in checkpoints directory! Skipping download.")

if not WEIGHTS_PATH:
    print(f"Weights '{WEIGHT_FILE}' not found locally. Downloading from Google Drive...")
    os.makedirs('checkpoints', exist_ok=True)
    !pip install --upgrade gdown
    # Download the whole folder containing GoPro.pth, RealBlur_J.pth, etc.
    !gdown --folder 1M-Sc_u97vTQskfO6VMzPghb8LnerP8Id -O checkpoints/
    
    if os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
        WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    else:
        # Fallback to GoPro.pth if RealBlur_J.pth was not downloaded successfully
        WEIGHTS_PATH = "./checkpoints/GoPro.pth"

print(f"Selected weights path for inference: {WEIGHTS_PATH}")

## Step 4: Add Blurred Images
Make sure you place your 15 blurred test images inside the `test_images/` folder.

**Tip**: If you commit your 15 test images to your GitHub repository under `test_images/` before running, they will be cloned automatically in Step 1 and you won't need to upload them manually!

In [ ]:
os.makedirs('test_images', exist_ok=True)
print(f"Images inside test_images/: {os.listdir('test_images')}")

## Step 5: Run Deblurring Inference
We will run `inference_custom.py` on our images. 

### Parameter Guide:
- `--input_dir`: Folder containing blurred images (default `./test_images`).
- `--output_dir`: Folder to save restored images (default `./results`).
- `--weights`: Weight file path. Automatically determined in Step 3.
- `--chunk_size`: Batch size of patches sent to the GPU at once. **Decrease this number (e.g., to 4 or 2) if you run out of memory. Increase it (e.g., to 16 or 32) if you have ample VRAM to speed up inference.**
- `--inference_mode`: `normal` (standard) or `eval` (DO-Conv folded models).
- `--limit`: Limit the number of images to process (default -1: process all). Set to a small number (e.g. 3) to test quickly without waiting for all images to complete.

In [ ]:
!python inference_custom.py \
    --input_dir ./test_images \
    --output_dir ./results \
    --weights {WEIGHTS_PATH} \
    --win_size 256 \
    --chunk_size 8 \
    --inference_mode normal \
    --limit 3

## Step 6: Side-by-Side Visualization
Let's visualize the original blurred image vs the restored deblurred image.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

blurred_dir = './test_images'
restored_dir = './results'

restored_files = [f for f in sorted(os.listdir(restored_dir)) if f.endswith('_deblurred.png')]
if restored_files:
    # Visualize the first restored image
    restored_name = restored_files[0]
    original_name = restored_name.replace('_deblurred.png', '')
    
    # Match original blurred image by prefix
    original_file = None
    for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp']:
        potential_path = os.path.join(blurred_dir, original_name + ext)
        if os.path.exists(potential_path):
            original_file = original_name + ext
            break
            
    if original_file:
        print(f"Comparing: {original_file} vs {restored_name}")
        img_blur = Image.open(os.path.join(blurred_dir, original_file))
        img_restore = Image.open(os.path.join(restored_dir, restored_name))
        
        # For very large images, display a scaled down version for plotting
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(img_blur)
        axes[0].set_title('Original Blurred Image', fontsize=16)
        axes[0].axis('off')
        
        axes[1].imshow(img_restore)
        axes[1].set_title('Restored (MISCFilter) Image', fontsize=16)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Could not find matching original image for {restored_name} in {blurred_dir}")
else:
    print("No deblurred results found in './results'. Make sure Step 5 completed successfully.")

## Step 7: Zip & Download Results
Run this cell to zip the output results and download them directly if you are in Google Colab.

In [ ]:
!zip -r deblurred_results.zip results/

try:
    from google.colab import files
    files.download('deblurred_results.zip')
    print("Triggered Colab download dialog.")
except ImportError:
    print("Not running on Google Colab. Please manually download 'deblurred_results.zip' using the sidebar file browser.")